# CountSort (sortare prin numărare)

Toate sortările de până acum (MergeSort, HeapSort, QuickSort) **comparau** elemente, deci făceau Ω(n log n) comparații (C6). CountSort nu compară nimic: **numără** de câte ori apare fiecare valoare. Funcționează doar pentru **întregi dintr-un interval mic** `[0, k)`.

Proprietăți:
- complexitate **O(n + k)** în toate cazurile (k = max + 1)
- memorie suplimentară **O(n + k)**
- **stabil** (varianta cu sume parțiale, cu parcurgere de la final)

## 1. Sortare stabilă

O sortare este **stabilă** dacă elementele cu chei egale își păstrează ordinea relativă din vectorul inițial.

Contează când sortăm **înregistrări**, nu doar numere: dacă sortăm studenții după notă, doi studenți cu aceeași notă sunt totuși persoane diferite.

**Sortare după mai multe criterii**: sortăm stabil întâi după criteriul **cel mai puțin important**, apoi după următorul, ..., iar la final după cel **mai important**. Stabilitatea garantează că, la egalitate pe criteriul curent, rămâne ordinea dată de criteriile anterioare.

`sorted` și `list.sort` din Python sunt stabile. În C++, `std::sort` **nu** e stabil, dar `std::stable_sort` este.

In [ ]:
studenti = [("Florin", 9), ("Bogdan", 7), ("Ana", 9), ("Elena", 7), ("Dana", 10), ("Cristi", 9)]

# vrem dupa nota descrescator, iar la egalitate alfabetic dupa nume
pas1 = sorted(studenti, key=lambda s: s[0])                  # intai dupa criteriul mai putin important
pas2 = sorted(pas1, key=lambda s: s[1], reverse=True)        # apoi (stabil) dupa cel important
for nume, nota in pas2:
    print(nota, nume)

## 2. CountSort naiv

Numărăm aparițiile: `f[x]` = de câte ori apare valoarea x. Apoi parcurgem `f` și scriem fiecare valoare `x` de `f[x]` ori.

Merge pentru **numere**, dar nu și pentru înregistrări: pierdem informația despre cui îi aparținea fiecare cheie.

In [ ]:
def countsort_naive(a):
    if not a:
        return []
    f = [0] * (max(a) + 1)
    for x in a:
        f[x] += 1
    result = []
    for x in range(len(f)):
        result.extend([x] * f[x])
    return result


print(countsort_naive([4, 1, 3, 4, 2, 0, 1, 4]))

## 3. CountSort cu sume parțiale (stabil)

1. Numărăm frecvențele în `f`.
2. **Sume parțiale**: `f[x] += f[x - 1]`. Acum `f[x]` = câte elemente sunt ≤ x, adică poziția (exclusiv) unde **se termină** blocul valorii x în vectorul sortat.
3. Parcurgem vectorul **de la final** și punem fiecare element `x` pe poziția `f[x] - 1`, apoi scădem `f[x]`.

De ce de la final? Ultimul element cu cheia x ajunge pe ultima poziție din blocul lui x, penultimul înaintea lui etc., deci ordinea relativă se păstrează ⇒ **stabil**.

In [ ]:
# countsort stabil, key(x) trebuie sa fie intreg din [0, k)
def countsort(a, key=lambda x: x):
    if not a:
        return []
    k = max(key(x) for x in a) + 1
    f = [0] * k
    for x in a:                        # 1. frecvente
        f[key(x)] += 1
    for i in range(1, k):              # 2. sume partiale
        f[i] += f[i - 1]
    out = [None] * len(a)
    for x in reversed(a):              # 3. de la final => stabil
        out[f[key(x)] - 1] = x
        f[key(x)] -= 1
    return out


print(countsort([4, 1, 3, 4, 2, 0, 1, 4]))

Pas cu pas:

In [ ]:
def countsort_verbose(a):
    k = max(a) + 1
    f = [0] * k
    for x in a:
        f[x] += 1
    print("valori:       ", list(range(k)))
    print("frecvențe:    ", f)
    for i in range(1, k):
        f[i] += f[i - 1]
    print("sume parțiale:", f, "\n")
    out = [None] * len(a)
    for i in range(len(a) - 1, -1, -1):
        x = a[i]
        out[f[x] - 1] = x
        f[x] -= 1
        shown = " ".join("_" if y is None else str(y) for y in out)
        print(f"a[{i}] = {x} -> poziția {f[x]}:   {shown}")
    return out


countsort_verbose([4, 1, 3, 4, 2, 0, 1, 4])

Stabilitatea, pe înregistrări. Sortăm studenții după notă; la egalitate trebuie să rămână ordinea inițială:

In [ ]:
studenti = [("Ana", 9), ("Bogdan", 7), ("Cristi", 9), ("Dana", 10), ("Elena", 7), ("Florin", 9)]
print(countsort(studenti, key=lambda s: s[1]))

⚠️ Dacă la pasul 3 parcurgem de la **început** (cu aceleași formule), elementele egale ies **în ordine inversă**, deci sortarea nu mai e stabilă:

In [ ]:
def countsort_forward(a, key=lambda x: x):
    k = max(key(x) for x in a) + 1
    f = [0] * k
    for x in a:
        f[key(x)] += 1
    for i in range(1, k):
        f[i] += f[i - 1]
    out = [None] * len(a)
    for x in a:                        # de la inceput => nu mai e stabil
        out[f[key(x)] - 1] = x
        f[key(x)] -= 1
    return out


print(countsort_forward(studenti, key=lambda s: s[1]))

## 4. Limitări

- Complexitatea este O(n + k): dacă k e mare, pierdem. Pentru `[400000000, 5, 8]` am aloca și am parcurge un vector de frecvențe de 400 de milioane de elemente ca să sortăm 3 numere!
- Merge doar pe **chei întregi** (sau pe chei care pot fi transformate în întregi mici).
- **Numere negative**: translatăm cu minimul. Cheia devine `x - min`, deci k = max − min + 1.

Concluzie: CountSort e bun când **k = O(n)**. Pentru valori mari folosim **RadixSort** (`radixsort.ipynb`): câte un CountSort pe fiecare cifră.

In [ ]:
# merge si cu negative: cheia e x - min
def countsort_any(a):
    if not a:
        return []
    lo = min(a)
    return countsort(a, key=lambda x: x - lo)


print(countsort_any([3, -2, 7, 0, -5, 3, 1]))

## Testare

In [ ]:
import random

for _ in range(500):
    arr = [random.randint(0, 30) for _ in range(random.randint(0, 50))]
    assert countsort_naive(arr) == sorted(arr)
    assert countsort(arr) == sorted(arr)

    neg = [random.randint(-30, 30) for _ in range(random.randint(0, 50))]
    assert countsort_any(neg) == sorted(neg)

    # stabilitate: perechi (cheie, pozitia initiala), sorted e stabil deci comparam cu el
    recs = [(random.randint(0, 5), i) for i in range(random.randint(0, 50))]
    assert countsort(recs, key=lambda r: r[0]) == sorted(recs, key=lambda r: r[0])
print("toate testele au trecut")